## PLEASE IGNORE THIS — THE NEW CHARTS HAVE DIFFERENT TITLES/SUBTITLES.\n\n# Publish the San Antonio bicyclist charts to Datawrapper

Run the main analysis notebook first. This notebook creates three new Datawrapper charts from its output tables, moves them into the selected folder, publishes them and prints the public URLs. Re-running it creates new charts.

In [ ]:
from pathlib import Path
from getpass import getpass
import json
import pandas as pd
import requests

ROOT = Path.cwd()
OUT = ROOT / 'outputs'
DW_API = 'https://api.datawrapper.de/v3'
DW_TOKEN = getpass('Paste your Datawrapper API token: ')
if not DW_TOKEN.strip():
    raise RuntimeError('A Datawrapper API token is required.')
HEADERS = {'Authorization': f'Bearer {DW_TOKEN}'}

In [ ]:
# Find the nested destination folder: San Antonio > Sarup.
folders_response = requests.get(f'{DW_API}/folders', headers=HEADERS, timeout=120)
folders_response.raise_for_status()
folder_json = folders_response.json()

def flatten_folders(items):
    rows = []
    for item in items:
        for folder in item.get('folders', []):
            rows.append({'id': folder['id'], 'name': folder['name']})
            rows.extend(flatten_folders([folder]))
    return rows

folder_rows = flatten_folders(folder_json.get('list', []))
matches = [row for row in folder_rows if row['name'].strip().lower() == 'sarup']
if len(matches) == 1:
    folder_id = matches[0]['id']
    print(f"Using San Antonio folder {folder_id}.")
else:
    display(pd.DataFrame(folder_rows).sort_values(['name', 'id']))
    folder_id = int(input('Enter the Datawrapper folder ID: '))


In [ ]:
def publish_chart(title, chart_type, frame, metadata):
    create = requests.post(
        f'{DW_API}/charts', headers={**HEADERS, 'Content-Type': 'application/json'},
        json={'title': title, 'type': chart_type, 'folderId': folder_id, 'theme': 'datawrapper'},
        timeout=120
    )
    create.raise_for_status()
    chart_id = create.json()['id']
    csv_text = frame.to_csv(index=False)
    upload = requests.put(
        f'{DW_API}/charts/{chart_id}/data', headers={**HEADERS, 'Content-Type': 'text/csv'},
        data=csv_text.encode('utf-8'), timeout=120
    )
    upload.raise_for_status()
    edit = requests.patch(
        f'{DW_API}/charts/{chart_id}', headers={**HEADERS, 'Content-Type': 'application/json'},
        json={'metadata': metadata}, timeout=120
    )
    edit.raise_for_status()
    publish = requests.post(f'{DW_API}/charts/{chart_id}/publish', headers=HEADERS, timeout=180)
    publish.raise_for_status()
    return {'chart_id': chart_id, 'url': f'https://datawrapper.dwcdn.net/{chart_id}/1/'}

trend = pd.read_csv(OUT / 'datawrapper_sa_annual_trend.csv').rename(columns={'Crash Year':'year', 'serious_injuries':'serious injuries', 'deaths':'deaths'})
top10 = pd.read_csv(OUT / 'datawrapper_top10_city_rates.csv').rename(columns={'city':'city', 'serious_injury_rate_per_million':'serious injuries per million', 'death_rate_per_million':'deaths per million'})
scatter = pd.read_csv(OUT / 'datawrapper_bike_commute_vs_death_rate.csv').rename(columns={'bike_commute_pct':'bike commuters (%)', 'death_rate_per_million':'deaths per million'})

results = []
results.append(publish_chart(
    'San Antonio bicyclist deaths and serious injuries by year', 'd3-lines', trend,
    {'describe': {'intro':'Bicyclists killed or suspected seriously injured in San Antonio, 2016 through Sept. 1, 2026.', 'source-name':'TxDOT CRIS', 'source-url':'https://cris.dot.state.tx.us/'}, 'visualize': {'custom-colors': {'serious injuries':'#3478a4', 'deaths':'#e67e22'}}}
))
results.append(publish_chart(
    'Bicyclist death rates in Texas cities', 'd3-bars-grouped', top10,
    {'describe': {'intro':'The 10 highest bicyclist death rates among Texas places with populations of at least 65,000 and at least one bicyclist death, 2024 through Sept. 1, 2026.', 'source-name':'TxDOT CRIS and U.S. Census Bureau ACS 2024', 'source-url':'https://www.census.gov/data/developers/data-sets/acs-1year.html'}, 'visualize': {'custom-colors': {'serious injuries per million':'#3478a4', 'deaths per million':'#e67e22'}}}
))
results.append(publish_chart(
    'Bike commuting and bicyclist death rates in Texas cities', 'd3-scatter-plot', scatter,
    {'describe': {'intro':'Bike commuting share and bicyclist death rates among Texas places with at least one bicyclist death, 2024 through Sept. 1, 2026.', 'source-name':'TxDOT CRIS and U.S. Census Bureau ACS 2024', 'source-url':'https://www.census.gov/data/developers/data-sets/acs-1year.html'}, 'axes': {'x':'bike commuters (%)', 'y':'deaths per million', 'labels':'city'}}
))
pd.DataFrame(results)